# Restore a source-linked reactor comparison

Research question: which original statements and missing-temperature metadata support the selected PWR and BWR descriptions?

This example restores two catalogue entries. It does not compare operating-device performance or infer numerical temperatures from prose. Whole-entry classification remains open. Original publisher bodies are not included or relicensed.

Start Jupyter from the repository root, or set `ATLAS_REPOSITORY` to the complete accepted checkout. Use a Python 3.12+ kernel with the project's build dependencies and Node 24.21.0 on PATH. See [the research guide](README.md).


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import sys
from pathlib import Path

root = Path(os.environ.get("ATLAS_REPOSITORY", Path.cwd())).resolve()
if not (root / "metadata/evidence_profiles/profiles.json").is_file():
    raise ValueError("Set ATLAS_REPOSITORY to the complete accepted repository snapshot")
sys.path.insert(0, str(root))

example = root / "examples/research"
manifest: object = json.loads((example / "comparison-manifest.json").read_text(encoding="utf-8"))
if not isinstance(manifest, dict):
    raise ValueError("The retained manifest must be an object")
profile_sha256: object = manifest["profile_sha256"]
bundle_sha256: object = manifest["bundle_sha256"]
comparison_file: object = manifest["comparison_file"]
if not (
    isinstance(profile_sha256, str)
    and isinstance(bundle_sha256, str)
    and isinstance(comparison_file, str)
):
    raise ValueError("The retained manifest needs string digests and a comparison filename")
bundle_file = example / comparison_file
print("Expected whole-profile snapshot:", profile_sha256)
print("Expected original comparison file:", bundle_sha256)

## Restore through the public reader

The reader checks the complete original taxonomy, citations and profiles before rebuilding the ordered comparison with the same public model as the browser. It refuses a stale snapshot or any changed claims, sources, rights or compatibility result. Retain the snapshot and manifest you used; replacing both an input and its expected hash does not establish trusted provenance.


In [ ]:
from tools.research_comparison import restore_comparison
from tools.research_objects import object_fields, object_rows

comparison = restore_comparison(root, bundle_file, profile_sha256, bundle_sha256)
profile_rows = object_rows(comparison["profiles"])
source_rows = object_rows(comparison["sources"])
selection = object_fields(comparison["selection"])
print("Selection:", selection["entry_ids"])
print("Metadata licence:", comparison["metadata_license"])
print("Source rights:", comparison["source_rights"])
print("Restored profiles:", len(profile_rows))
print("Retained sources:", len(source_rows))

## Inspect original claims and locators

Keep the original statement beside its supporting source and page/section locator. Review dates and acquisition dates retain their original meanings. A null acquisition date stays unknown. A citation supports its stated scope; it does not supply complete independent entry acceptance.


In [ ]:
sources = {source["id"]: source for source in source_rows}
claim_rows = [
    {
        "entry_id": profile["entry_id"],
        "claim_id": claim["claim_id"],
        "original_statement": claim["original_statement"],
        "citation": citation,
        "source": sources[citation["source_id"]],
    }
    for profile in profile_rows
    for claim in object_rows(profile["claims"])
    for citation in [object_fields(claim["citation"])]
]
print(json.dumps(claim_rows, ensure_ascii=False, indent=2, allow_nan=False))

## Distinguish absence from an incompatible measured value

The original catalogue temperature descriptions have not been normalised with source-bound units, conditions and system boundaries. The example retains `not_reviewed` and `not_comparable`; it computes no temperature average or performance ranking.


In [ ]:
missing_parameters = [
    {"entry_id": profile["entry_id"], "parameter": parameter}
    for profile in profile_rows
    for parameter in object_rows(profile["parameters"])
    if parameter["state"] != "known"
]
print(json.dumps(missing_parameters, ensure_ascii=False, indent=2, allow_nan=False))
print(
    json.dumps(comparison["parameter_comparisons"], ensure_ascii=False, indent=2, allow_nan=False)
)

## Check repeatability and retain a research result

A second read must restore the same original comparison. The result below preserves every original profile and source, both explicit hashes, the ordered selection, the inspected claim rows and the missing-parameter observations. It is an in-memory result; the notebook writes no source, snapshot or accepted dataset. Save it to a new research file if needed and cite the original supporting works.


In [ ]:
repeated = restore_comparison(root, bundle_file, profile_sha256, bundle_sha256)
if repeated != comparison:
    raise ValueError("The repeated comparison did not reproduce the original result")
research_result = {
    "schema_version": "atlas-research-result-1.0.0",
    "profile_sha256": profile_sha256,
    "original_bundle_sha256": bundle_sha256,
    "selection": comparison["selection"],
    "comparison": comparison,
    "claim_rows": claim_rows,
    "missing_parameters": missing_parameters,
    "limitations": "Catalogue statements only; no inferred numeric temperature, complete entry acceptance or publisher-body redistribution",
}
result_json = json.dumps(
    research_result, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False
)
result_sha256 = hashlib.sha256(result_json.encode("utf-8")).hexdigest()
print("Repeated restoration agrees:", repeated == comparison)
print("Original statements retained:", len(claim_rows))
print("Missing numeric parameter records:", len(missing_parameters))
print("Deterministic research-result SHA-256:", result_sha256)